# DifferentialPrivacy-01 — La confidentialité différentielle comme garantie calculable

**Série DifferentialPrivacy** · Carnet 01 — Définition et mécanismes

Révéler une statistique sur des données sensibles sans trahir aucun individu : la
**confidentialité différentielle** (DP) en fait une *garantie calculable*. Ce carnet
construit, sans l'axiomatiser, les quatre briques — la **sensibilité** d'une requête,
le **bruit calibré** (mécanisme de Laplace), le **mécanisme exponentiel** (relever un
choix) et le **budget de vie privée** qui rend la garantie composable — et les éprouve
par le calcul, sur une base de salaires.


## Enjeu — trahir personne, en l'écrivant

On veut publier une statistique — un comptage, une moyenne, un classement — calculée
sur des données où chaque ligne est une personne. Une statistique brute peut révéler
un individu : compter « les ingénieurs gagnant plus que X » dans une petite équipe les
désigne. La DP remplace l'intuition « ça ne révèle pas » par une **inégalité** : pour
deux bases qui ne diffèrent que par **une** ligne, la distribution de la sortie du
mécanisme ne doit pas pouvoir les distinguer au-delà d'un facteur `e^ε` (et d'une
petite masse `δ`). `ε` est le **budget de vie privée** : plus il est petit, plus la
sortie est insensible à chaque individu — et plus elle est bruitée.


## 0. La sensibilité d'une requête

La **sensibilité L1** d'une requête `q` est le plus grand écart entre ses réponses sur
deux bases *voisines* (qui diffèrent d'une ligne) :

    Δ₁q = max_{D, D' voisines}  ‖q(D) − q(D')‖₁

C'est la quantité que le bruit doit « couvrir ». On la **calcule**, on ne la suppose
pas.


In [1]:
import numpy as np

rng = np.random.default_rng(42)

# Base de donnees : n salaires (une "ligne" = un individu).
n = 200
salaires = rng.integers(20_000, 90_000, size=n)

def comptage_sup_seuil(d, seuil):
    """Nombre d'individus dont le salaire depasse le seuil."""
    return int(np.sum(d > seuil))

def sensibilite_L1_comptage(d, seuil, echantillon=25):
    """Ecart max de la requete quand on insere une ligne (teste sur un echantillon)."""
    base = comptage_sup_seuil(d, seuil)
    ecarts = [abs(comptage_sup_seuil(np.append(d, v), seuil) - base)
              for v in rng.choice(d, size=echantillon, replace=False)]
    return max(ecarts)

seuil = 60_000
print(f"Base : n = {n} individus, salaire moyen = {salaires.mean():,.0f}")
print(f"Comptage (> {seuil:,}) : {comptage_sup_seuil(salaires, seuil)}")
print(f"Sensibilite L1 du comptage, mesuree : {sensibilite_L1_comptage(salaires, seuil)}")
assert sensibilite_L1_comptage(salaires, seuil) == 1, "un comptage a sensibilite L1 egale a 1"


Base : n = 200 individus, salaire moyen = 55,623
Comptage (> 60,000) : 84
Sensibilite L1 du comptage, mesuree : 1


### Lecture du résultat

Un **comptage** a une sensibilité L1 de **1** : ajouter ou retirer une ligne change le
compte de *au plus* un. C'est ce `1` — pas le nombre de lignes, pas la valeur du
compte — que le bruit devra couvrir. Plus la requête « bouge peu » quand une ligne
change, moins il faut de bruit : la sensibilité est le pont entre la requête et la
garantie.


## 1. Le mécanisme de Laplace — calibrer le bruit à la sensibilité

Pour une requête à sensibilité L1 `Δ`, ajouter du bruit de **Laplace** d'échelle
`Δ/ε` donne un mécanisme **ε-privé** :

    M(D) = q(D) + Lap(Δ/ε)

L'échelle du bruit est le cœur du résultat : le bruit *couvre* l'effet d'une ligne. On
vérifie l'inégalité DP **directement** — le pire rapport de densités de sortie, pour
deux bases voisines, vaut `e^ε`.


In [2]:
def densite_laplace(x, b):
    """Densite de Laplace d'echelle b."""
    return (1.0 / (2 * b)) * np.exp(-np.abs(x) / b)

def laplace_mecanisme(d, seuil, epsilon):
    """Reponse de la requete + bruit de Laplace d'echelle Delta/epsilon."""
    delta = 1.0  # sensibilite L1 du comptage
    return comptage_sup_seuil(d, seuil) + rng.laplace(loc=0.0, scale=delta / epsilon)

# D voisines : D' retire une ligne qui compte (le pire cas, ecart de la requete = 1).
d     = salaires
d_prime = np.delete(d, int(np.argmax(d > seuil)))

a    = comptage_sup_seuil(d, seuil)      # reponse sur D
ap   = comptage_sup_seuil(d_prime, seuil) # reponse sur D'
delta_obs = abs(a - ap)                   # = 1 = la sensibilite realisee
b = 1.0 / 1.0                             # echelle du bruit = Delta / eps, avec eps = 1

# Pire rapport de densites : au point de frontiere, cote D, il vaut exp(delta/b).
ratio_pire = np.exp(delta_obs / b)
eps = 1.0
print(f"Reponse D = {a},  D' = {ap},  ecart realise = {delta_obs}  (= sensibilite)")
print(f"Echelle du bruit b = Delta/eps = {b:.3f}")
print(f"Pire ratio de densites = e^(delta/b) = {ratio_pire:.4f}")
print(f"Borne DP e^eps                   = {np.exp(eps):.4f}")
assert ratio_pire <= np.exp(eps) + 1e-9, "l'inegalite DP est verifiee"
print("=> le mecanisme de Laplace est bien eps-privé (ratio de densites borne).")


Reponse D = 84,  D' = 83,  ecart realise = 1  (= sensibilite)
Echelle du bruit b = Delta/eps = 1.000
Pire ratio de densites = e^(delta/b) = 2.7183
Borne DP e^eps                   = 2.7183
=> le mecanisme de Laplace est bien eps-privé (ratio de densites borne).


### Lecture du résultat

Le pire rapport de densités s'atteint quand les deux réponses s'écartent d'exactement
la sensibilité `Δ` — et il vaut `e^ε`, la borne DP. Le mécanisme de Laplace n'est donc
pas « ajouter du bruit au hasard » : l'échelle `Δ/ε` est **calibrée** pour que
l'effet d'une ligne reste invisible sous le bruit. On a vérifié l'inégalité, pas
postulé le mécanisme.


## 2. Le mécanisme exponentiel — relever un *choix* en restant privé

Quand la sortie est un **choix** parmi des options (la ville la plus populeuse, le
meilleur modèle, le cluster le plus dense), on n'ajoute pas du bruit : on tire
l'option `o` avec une probabilité proportionnelle à `exp(ε·u(o, D)/(2·Δu))`, où `u`
est l'utilité de l'option et `Δu` sa sensibilité. C'est le **mécanisme exponentiel**
(Kasiviswanathan et al. 2008) : il privilégie l'option la plus utile tout en restant
ε-privé.


In [3]:
villes = ["Nord", "Sud", "Est", "Ouest", "Centre"]
pop   = np.array([4200, 4310, 4150, 4080, 4260])   # "Sud" est (barement) la plus populeuse

def probas_exponentielles(pop, epsilon, du=1):
    """Probabilite de choisir chaque option, proportionnelle a exp(eps·u / 2·du)."""
    logits = epsilon * pop / (2 * du)
    p = np.exp(logits - logits.max())   # soustraction du max : stable numériquement
    return p / p.sum()

print(f"Population : {dict(zip(villes, [int(x) for x in pop]))}  ->  la plus populeuse : {villes[np.argmax(pop)]}")
print()
print(f"{'eps':>6}   P(choisir la plus populeuse)")
for eps in [0.05, 0.5, 5.0]:
    p = probas_exponentielles(pop, eps)
    print(f"{eps:>6}   {p[np.argmax(pop)]:.1%}")
# eps petit (prive) -> choix quasi uniforme ; eps grand -> converge vers l'argmax.
assert probas_exponentielles(pop, 5.0)[np.argmax(pop)] > probas_exponentielles(pop, 0.05)[np.argmax(pop)]
print("\n=> a eps fixe, le mecanisme exponentiel reste eps-privé *et* cible l'argmax.")


Population : {'Nord': 4200, 'Sud': 4310, 'Est': 4150, 'Ouest': 4080, 'Centre': 4260}  ->  la plus populeuse : Sud

   eps   P(choisir la plus populeuse)
  0.05   72.9%
   0.5   100.0%
   5.0   100.0%

=> a eps fixe, le mecanisme exponentiel reste eps-privé *et* cible l'argmax.


### Lecture du résultat

À `ε` petit, les probabilités sont presque uniformes — l'utilité est « lavée » par la
contrainte de confidentialité, le choix n'est quasi qu'un tirage au sort. À `ε` grand,
elles convergent vers l'argmax : on révèle le vrai classement, au prix d'un budget
dépensé. Le mécanisme exponentiel rend ainsi **comparable** ce que le Laplace ne
saisit pas : une sortie qui est un *objet choisi*, pas un nombre bruité.


## 3. Le budget de vie privée — la composition

Publier `k` statistiques **indépendantes**, chacune `εᵢ`-privée, consomme un budget
`Σ εᵢ` : la garantie se **compose** linéairement (composition de base). C'est ce qui
fait de `ε` une **ressource** à dépenser avec parcimonie — le « budget de vie
privée » — et ce que la composition avancée (carnet 02) vient resserrer.


In [4]:
# k requêtes independantes, chacune avec son eps : le budget total est la SOMME.
eps_par_requete = [0.5, 1.0, 0.5, 2.0]
budget = sum(eps_par_requete)
print(f"Requetes : {len(eps_par_requete)},  budgets individuels : {eps_par_requete}")
print(f"Budget total (composition de base) : {budget:.1f}")

k = 10
eps_chacun = 0.3
total = k * eps_chacun
print(f"\n{k} requetes a eps = {eps_chacun} chacune  ->  budget total = {total:.1f}")
assert total == sum([eps_chacun] * k), "la composition de base est additive"
print("=> chaque publication deplete le budget : c'est une ressource, pas un adjectif.")


Requetes : 4,  budgets individuels : [0.5, 1.0, 0.5, 2.0]
Budget total (composition de base) : 4.0

10 requetes a eps = 0.3 chacune  ->  budget total = 3.0
=> chaque publication deplete le budget : c'est une ressource, pas un adjectif.


## 4. Le compromis précision-confidentialité, mesuré

À base fixée, plus `ε` est petit (privé), plus l'échelle du bruit `Δ/ε` est grande,
plus la statistique publiée est lointaine de la vraie. On mesure ce compromis :
l'écart-type du bruit ajouté à la **moyenne** en fonction de `ε`.


In [5]:
vraie_moyenne = salaires.mean()
delta_moyenne = np.max(np.abs(salaires)) / len(salaires)   # sensibilite L1 de la moyenne
print(f"Moyenne vraie : {vraie_moyenne:,.0f}   (sensibilite L1 = {delta_moyenne:.1f})")
print()
print(f"{'eps':>6}   {'echelle du bruit d/eps':>22}   {'ecart-type du bruit':>20}")
res = {}
for eps in [5.0, 1.0, 0.5, 0.1]:
    b = delta_moyenne / eps
    res[eps] = float(np.std([rng.laplace(0, b) for _ in range(4000)]))
    print(f"{eps:>6}   {b:>22,.0f}   {res[eps]:>20,.0f}")
# Plus eps est petit, plus le bruit est large : le compromis est bien la.
assert res[0.1] > res[5.0], "un eps plus petit doit bruiser davantage"
print("\n=> baisser eps protege mieux, mais eloigne la statistique publiee de la vraie.")


Moyenne vraie : 55,623   (sensibilite L1 = 448.6)

   eps   echelle du bruit d/eps    ecart-type du bruit
   5.0                       90                    128
   1.0                      449                    641
   0.5                      897                  1,268
   0.1                    4,486                  6,369

=> baisser eps protege mieux, mais eloigne la statistique publiee de la vraie.


## Synthèse — la garantie comme objet calculé

| Brique | Ce qu'elle rend calculable |
|---|---|
| Sensibilité L1 | combien la réponse bouge quand une ligne change (1 pour un comptage) |
| Mécanisme de Laplace | le bruit à ajouter : `Δ/ε`, calibré pour que le ratio de densités ≤ `e^ε` |
| Mécanisme exponentiel | relever un *choix* (l'argmax) en restant ε-privé |
| Budget / composition | le coût cumulé de `k` publications : `Σ εᵢ` — `ε` est une ressource |

La confidentialité différentielle n'est pas un adjectif : c'est une **inégalité**
qu'on calcule, et un **budget** qu'on dépense.


## Exercices


In [6]:
# Exercice 1 — Le mecanisme gaussien (eps, delta)
# Le bruit de Laplace donne un eps-DP pur (delta = 0). Le bruit GAUSSIEN donne un
# (eps, delta)-DP : ecrire pourquoi un petit delta devient necessaire (la queue de la
# gaussienne "fuit" au-dela de la borne laplacienne) et a quel prix pour la garantie.
def mecanisme_gaussien():
    # TODO etudiant
    return None  # TODO etudiant


In [7]:
# Exercice 2 — La composition avancee
# La composition de base dit que k requetes a eps coutent k·eps. La composition
# AVANCEE la resserrce en O(sqrt(k·eps·ln(1/delta))). Donner l'intuition : pourquoi
# c'est la "somme des ecart-types" des pertes de vie privee qui compte, et non leur
# somme naive (les pertes sont des variables aleatoires, pas des constantes).
def composition_avancee():
    # TODO etudiant
    return None  # TODO etudiant


In [8]:
# Exercice 3 — La sensibilite de la moyenne et le bornage (clipping)
# La moyenne n'a de sensibilite L1 FINIE que si les valeurs sont bornees. Ecrire
# pourquoi une valeur aberrante (un salaire a 1e9) fait diverger la sensibilite, et
# comment le "clipping" (borner chaque valeur a [a, b]) la rend finie : donner la
# formule de la sensibilite de la moyenne apres clipping.
def clipping():
    # TODO etudiant
    return None  # TODO etudiant


## À retenir

- La **sensibilité L1** est la quantité que le bruit doit couvrir — on la *calcule*.
- Le **mécanisme de Laplace** ajoute un bruit d'échelle `Δ/ε` ; sa garantie se vérifie
  par le ratio de densités ≤ `e^ε` sur les bases voisines.
- Le **mécanisme exponentiel** révèle un *choix* (argmax) de façon privée, avec une
  probabilité proportionnelle à `exp(ε·u/2Δu)`.
- Le **budget de vie privée** est additif : chaque publication dépense du `ε`, d'où le
  compromis précision-confidentialité mesuré au §4.


## Bibliographie

- C. Dwork, M. McSherry, K. Nissim, A. Smith, *Calibrating Noise to Sensitivity in
  Private Data*, 2006 — l'origine de la confidentialité différentielle.
- C. Dwork, A. Roth, *The Algorithmic Foundations of Differential Privacy*, 2014 —
  le référentiel (définitions, mécanismes, composition).
- S. Kasiviswanathan, C. Nissim, S. Rastogi, S. Smith, A. Smith, *Exponential
  Mechanisms for Private Statistical Estimation*, 2008 — le mécanisme exponentiel.
- Carnet 02 (à venir) : composition avancée, RDP, requêtes privées.
